In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test\pilot_01")
csv_path = project_dir / "inspection" / "07_community_reports_all.csv"

reports = pd.read_csv(csv_path)

print("Number of community reports:", len(reports))
print("Columns:", reports.columns.tolist())

reports[
    ["community", "level", "title", "summary", "rank", "size"]
].head(20)

Number of community reports: 62
Columns: ['id', 'human_readable_id', 'community', 'level', 'parent', 'children', 'title', 'summary', 'full_content', 'rank', 'rating_explanation', 'findings', 'full_content_json', 'period', 'size']


,community,level,title,summary,rank,size
0,60,2,Austrian Federal Government and National PV De...,This community centers on the Austrian Federal...,8.0,2
1,61,2,Austrian Photovoltaic Strategy Leadership and ...,This community centers around Austria's nation...,8.5,11
2,16,1,E-Control and Austria's Grid Connection Action...,"This community centers on E-Control, Austria’s...",8.5,6
3,17,1,Austria’s Photovoltaic and Energy Transition S...,This community comprises key Austrian institut...,8.0,9
4,18,1,Austrian Federal Ministry for Climate Action a...,This community centers around Austria's Federa...,8.5,4
5,19,1,Austrian Renewable Expansion Act and AGRI-PV-A...,This community centers on the implementation o...,8.5,4
6,20,1,Austria's Photovoltaic Sector and Energy Marke...,This community centers on Austria’s rapidly ev...,8.5,12
7,21,1,Austrian Biodiversitäts-Solarpark and Freifläc...,This community encompasses Austria's effort to...,8.5,2
8,22,1,Austrian Solarindustrie and PV-Wertschöpfungsk...,The Austrian solar industry (Solarindustrie) a...,8.0,2
9,23,1,Austrian Unternehmen and the EU-Taxonomy Susta...,This community centers around Austrian Unterne...,8.0,3


In [2]:
top_ranked_reports = reports.nlargest(5, "rank")

largest_reports = reports.nlargest(5, "size")

random_reports = reports.sample(
    n=7,
    random_state=42
)

community_report_sample = (
    pd.concat(
        [
            top_ranked_reports,
            largest_reports,
            random_reports,
        ]
    )
    .drop_duplicates(subset="id")
    .reset_index(drop=True)
)

print("Community-report audit sample size:", len(community_report_sample))

community_report_sample[
    [
        "community",
        "level",
        "title",
        "summary",
        "full_content",
        "rank",
        "size",
    ]
]

Community-report audit sample size: 12


,community,level,title,summary,full_content,rank,size
0,52,1,Austria Renewable Energy & Photovoltaic Transi...,"The Austrian renewable energy community, ancho...",# Austria Renewable Energy & Photovoltaic Tran...,9.2,34
1,0,0,"Austria's Photovoltaic Energy Transition: BMK,...",This community centers on Austria’s ambitious ...,# Austria's Photovoltaic Energy Transition: BM...,9.2,19
2,4,0,EU Climate Policy Community: European Commissi...,This community centers on the European Union’s...,# EU Climate Policy Community: European Commis...,9.2,12
3,15,0,Austria’s Photovoltaic and Renewable Energy Tr...,This report examines the interconnected networ...,# Austria’s Photovoltaic and Renewable Energy ...,9.1,62
4,24,1,"European Union Green Policy Community: EU, Mem...",This report covers the core community at the h...,"# European Union Green Policy Community: EU, M...",9.0,4
5,5,0,Austrian National Photovoltaic Transition and ...,This community centers around Austria’s ambiti...,# Austrian National Photovoltaic Transition an...,9.0,34
6,11,0,Austria Photovoltaic Expansion: Key Entities a...,This community comprises entities central to t...,# Austria Photovoltaic Expansion: Key Entities...,8.5,21
7,2,0,Austria’s Photovoltaic and Renewable Energy Co...,This community represents the interconnected n...,# Austria’s Photovoltaic and Renewable Energy ...,9.0,20
8,10,0,Austrian PV Industry and Research Innovation C...,This community comprises the Austrian PV (phot...,# Austrian PV Industry and Research Innovation...,8.5,7
9,60,2,Austrian Federal Government and National PV De...,This community centers on the Austrian Federal...,# Austrian Federal Government and National PV ...,8.0,2


In [3]:
community_report_review = community_report_sample[
    [
        "id",
        "community",
        "level",
        "title",
        "summary",
        "full_content",
        "rank",
        "size",
    ]
].copy()

community_report_review["source_grounding"] = ""
community_report_review["primary_classification"] = ""
community_report_review["unsupported_language"] = ""
community_report_review["inherited_graph_error"] = ""
community_report_review["new_llm_claim"] = ""
community_report_review["audit_notes"] = ""

community_report_review.head()

,id,community,level,title,summary,full_content,rank,size,source_grounding,primary_classification,unsupported_language,inherited_graph_error,new_llm_claim,audit_notes
0,c3a0e1733a128a4eff4ee876e5a7160fe747b97c8e4b33...,52,1,Austria Renewable Energy & Photovoltaic Transi...,"The Austrian renewable energy community, ancho...",# Austria Renewable Energy & Photovoltaic Tran...,9.2,34,,,,,,
1,f815fbd9072ebfce437071efe904cb9cb9c8e84e647b95...,0,0,"Austria's Photovoltaic Energy Transition: BMK,...",This community centers on Austria’s ambitious ...,# Austria's Photovoltaic Energy Transition: BM...,9.2,19,,,,,,
2,594f38cb8cbbd564029dbecb637ae5fed7e0cc15a20260...,4,0,EU Climate Policy Community: European Commissi...,This community centers on the European Union’s...,# EU Climate Policy Community: European Commis...,9.2,12,,,,,,
3,d07475061a3499ad27b84f71992f8475cbd514c3d7eebb...,15,0,Austria’s Photovoltaic and Renewable Energy Tr...,This report examines the interconnected networ...,# Austria’s Photovoltaic and Renewable Energy ...,9.1,62,,,,,,
4,40f1882a2d072ab7ac07360120fab7a734fb2b72f81903...,24,1,"European Union Green Policy Community: EU, Mem...",This report covers the core community at the h...,"# European Union Green Policy Community: EU, M...",9.0,4,,,,,,


In [6]:
def add_report_review(
    row_number,
    grounding,
    classification,
    unsupported_language,
    inherited_error,
    new_claim,
    notes,
):
    community_report_review.loc[row_number, "source_grounding"] = grounding
    community_report_review.loc[row_number, "primary_classification"] = classification
    community_report_review.loc[row_number, "unsupported_language"] = unsupported_language
    community_report_review.loc[row_number, "inherited_graph_error"] = inherited_error
    community_report_review.loc[row_number, "new_llm_claim"] = new_claim
    community_report_review.loc[row_number, "audit_notes"] = notes

    

In [7]:
add_report_review(
    row_number=0,
    grounding="PARTIAL",
    classification="CONTAINS_LLM_INTRODUCED_CLAIMS",
    unsupported_language="European frontrunner; international leadership; model and template for other countries.",
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The report combines many source-supported topics, but adds broad claims "
        "about leadership, systemic impact and international significance that "
        "are not explicitly established by the source document."
    ),
)

add_report_review(
    row_number=1,
    grounding="PARTIAL",
    classification="CONTAINS_LLM_INTRODUCED_CLAIMS",
    unsupported_language="BMK and E-Control spearhead the transition; complete governance model.",
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "BMK and E-Control are relevant entities, but the report expands their "
        "roles and creates a stronger governance and causal narrative than the "
        "source supports. The statement about a 13th edition also appears suspicious."
    ),
)

add_report_review(
    row_number=2,
    grounding="LOW",
    classification="CONTAINS_LLM_INTRODUCED_CLAIMS",
    unsupported_language="EU orchestrates the transition; stands at the apex; worldwide leadership.",
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The source mentions EU policies, but the report relies heavily on general "
        "knowledge about the EU. It extends beyond the Austrian PV Strategy and "
        "introduces institutional and leadership claims."
    ),
)

add_report_review(
    row_number=3,
    grounding="PARTIAL",
    classification="CONTAINS_LLM_INTRODUCED_CLAIMS",
    unsupported_language="Austria is a leader, model and at the forefront of Europe's renewable transition.",
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "Most topics originate from the source, including PV policy, legislation, "
        "research and multilevel governance. However, the report combines them "
        "into a broad success and leadership narrative not explicitly stated."
    ),
)

add_report_review(
    row_number=4,
    grounding="PARTIAL",
    classification="UNSUPPORTED_ELABORATION",
    unsupported_language="EU stands at the apex; tightly integrated legal ecosystem; direct enforceable impact.",
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The policies and directives are relevant, but the report substantially "
        "elaborates their legal hierarchy, enforcement and economic effects beyond "
        "what the Austrian source document explains."
    ),
)

add_report_review(
    row_number=5,
    grounding="PARTIAL",
    classification="CAUSAL_GENERALIZATION",
    unsupported_language="Legal frameworks directly produce adoption, sovereignty, acceptance and economic growth.",
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The report accurately identifies several major policy and technical themes, "
        "but repeatedly converts strategic recommendations into established causal "
        "effects and presents intended measures as if they were already implemented."
    ),
)

community_report_review.loc[
    :5,
    [
        "community",
        "title",
        "source_grounding",
        "primary_classification",
        "inherited_graph_error",
        "new_llm_claim",
        "audit_notes",
    ],
]

,community,title,source_grounding,primary_classification,inherited_graph_error,new_llm_claim,audit_notes
0,52,Austria Renewable Energy & Photovoltaic Transi...,PARTIAL,CONTAINS_LLM_INTRODUCED_CLAIMS,YES,YES,The report combines many source-supported topi...
1,0,"Austria's Photovoltaic Energy Transition: BMK,...",PARTIAL,CONTAINS_LLM_INTRODUCED_CLAIMS,YES,YES,"BMK and E-Control are relevant entities, but t..."
2,4,EU Climate Policy Community: European Commissi...,LOW,CONTAINS_LLM_INTRODUCED_CLAIMS,YES,YES,"The source mentions EU policies, but the repor..."
3,15,Austria’s Photovoltaic and Renewable Energy Tr...,PARTIAL,CONTAINS_LLM_INTRODUCED_CLAIMS,YES,YES,"Most topics originate from the source, includi..."
4,24,"European Union Green Policy Community: EU, Mem...",PARTIAL,UNSUPPORTED_ELABORATION,YES,YES,"The policies and directives are relevant, but ..."
5,5,Austrian National Photovoltaic Transition and ...,PARTIAL,CAUSAL_GENERALIZATION,YES,YES,The report accurately identifies several major...


In [8]:
add_report_review(
    row_number=6,
    grounding="PARTIAL",
    classification="CAUSAL_GENERALIZATION",
    unsupported_language=(
        "PV expansion actively shapes continental electricity markets; "
        "incentives ensure long-term profitability."
    ),
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The source discusses PV installations, grids, incentives and electricity "
        "markets. However, the report presents possible effects as established "
        "causal outcomes and incorrectly describes PV installations as an "
        "ecosystem of organizations and actors."
    ),
)

add_report_review(
    row_number=7,
    grounding="PARTIAL",
    classification="CAUSAL_GENERALIZATION",
    unsupported_language=(
        "Agri-PV enhances rural economic resilience; biodiversity systems create "
        "ecological restoration; active customers increase grid resilience."
    ),
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The report uses genuine source concepts, including Agri-PV, biodiversity, "
        "active customers and skilled workers. It nevertheless turns intended "
        "benefits and policy objectives into demonstrated outcomes."
    ),
)

add_report_review(
    row_number=8,
    grounding="PARTIAL",
    classification="UNSUPPORTED_ELABORATION",
    unsupported_language=(
        "Research-industry collaboration forms the backbone of competitiveness; "
        "Made in Austria boosts exports and produces positive economic spillovers."
    ),
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "Research, innovation, domestic production and the PV industry are discussed "
        "in the source. The report elaborates their relationships and economic effects "
        "beyond the evidence contained in the strategy."
    ),
)

add_report_review(
    row_number=9,
    grounding="PARTIAL",
    classification="CONTAINS_LLM_INTRODUCED_CLAIMS",
    unsupported_language=(
        "Formal government mandate, enforcement, procurement compliance, "
        "benchmarking and reputational leadership."
    ),
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "The source discusses using suitable federal buildings for PV deployment. "
        "The report expands this limited recommendation into a formal governance, "
        "enforcement and leadership system that is not established by the source."
    ),
)

add_report_review(
    row_number=10,
    grounding="PARTIAL",
    classification="CAUSAL_GENERALIZATION",
    unsupported_language=(
        "Agri-PV prevents market domination, guarantees food security, creates "
        "economic viability and demonstrates strong rural acceptance."
    ),
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "EAG support and the dual agricultural and electricity function of Agri-PV "
        "are source-supported. Claims about market structure, food security, economic "
        "success and public acceptance go beyond the source."
    ),
)

add_report_review(
    row_number=11,
    grounding="LOW",
    classification="CONTAINS_LLM_INTRODUCED_CLAIMS",
    unsupported_language=(
        "ÖGUT has a central operational role; its methods were formally evaluated, "
        "validated and adopted by government."
    ),
    inherited_error="YES",
    new_claim="YES",
    notes=(
        "ÖGUT and its social-innovation publication appear mainly as a referenced "
        "source. The community report transforms this citation into an active "
        "institutional role and invents evaluation, validation and adoption processes."
    ),
)

community_report_review.loc[
    6:11,
    [
        "community",
        "title",
        "source_grounding",
        "primary_classification",
        "inherited_graph_error",
        "new_llm_claim",
        "audit_notes",
    ],
]

,community,title,source_grounding,primary_classification,inherited_graph_error,new_llm_claim,audit_notes
6,11,Austria Photovoltaic Expansion: Key Entities a...,PARTIAL,CAUSAL_GENERALIZATION,YES,YES,"The source discusses PV installations, grids, ..."
7,2,Austria’s Photovoltaic and Renewable Energy Co...,PARTIAL,CAUSAL_GENERALIZATION,YES,YES,"The report uses genuine source concepts, inclu..."
8,10,Austrian PV Industry and Research Innovation C...,PARTIAL,UNSUPPORTED_ELABORATION,YES,YES,"Research, innovation, domestic production and ..."
9,60,Austrian Federal Government and National PV De...,PARTIAL,CONTAINS_LLM_INTRODUCED_CLAIMS,YES,YES,The source discusses using suitable federal bu...
10,19,Austrian Renewable Expansion Act and AGRI-PV-A...,PARTIAL,CAUSAL_GENERALIZATION,YES,YES,EAG support and the dual agricultural and elec...
11,30,ÖGUT and Social Innovation Methods in Austria’...,LOW,CONTAINS_LLM_INTRODUCED_CLAIMS,YES,YES,ÖGUT and its social-innovation publication app...


In [9]:
reviewed_reports = community_report_review[
    community_report_review["primary_classification"] != ""
]

print("Reviewed community reports:", len(reviewed_reports))

print()
print("Primary classifications:")
print(reviewed_reports["primary_classification"].value_counts())

print()
print("Source grounding:")
print(reviewed_reports["source_grounding"].value_counts())

print()
print("Inherited graph errors:")
print(reviewed_reports["inherited_graph_error"].value_counts())

print()
print("Reports containing new LLM claims:")
print(reviewed_reports["new_llm_claim"].value_counts())

Reviewed community reports: 12

Primary classifications:
primary_classification
CONTAINS_LLM_INTRODUCED_CLAIMS    6
CAUSAL_GENERALIZATION             4
UNSUPPORTED_ELABORATION           2
Name: count, dtype: int64

Source grounding:
source_grounding
PARTIAL    10
LOW         2
Name: count, dtype: int64

Inherited graph errors:
inherited_graph_error
YES    12
Name: count, dtype: int64

Reports containing new LLM claims:
new_llm_claim
YES    12
Name: count, dtype: int64


# Stage 6 — Audit of GraphRAG Community Reports

## Objective

This stage evaluated whether GraphRAG's community reports faithfully summarize the Austrian Photovoltaic Strategy or introduce additional interpretations that are not supported by the original source.

Community reports are generated by an LLM from previously extracted entities and relationships. They are therefore not direct source text and may inherit errors from earlier pipeline stages.

## Dataset and sampling

GraphRAG generated **62 community reports**.

A sample of **12 reports** was reviewed. The sample combined:

- five reports with the highest GraphRAG rank,
- five reports representing the largest communities,
- seven randomly selected reports using `random_state=42`,
- removal of duplicated selections.

This sampling method intentionally includes prominent and structurally important reports. The resulting proportions describe the reviewed sample and should not automatically be generalized to all 62 reports.

## Classification results

| Classification | Number | Percentage |
|---|---:|---:|
| Contains LLM-introduced claims | 6 | 50.0% |
| Causal generalization | 4 | 33.3% |
| Unsupported elaboration | 2 | 16.7% |
| Accurate synthesis | 0 | 0.0% |
| **Total** | **12** | **100%** |

## Source grounding

| Source grounding | Number | Percentage |
|---|---:|---:|
| Partial | 10 | 83.3% |
| Low | 2 | 16.7% |
| High | 0 | 0.0% |
| **Total** | **12** | **100%** |

All 12 reviewed reports inherited problems from the extracted graph, and all 12 introduced at least some new LLM-generated interpretation.

## Main observed problems

### 1. Source-supported topics were converted into stronger claims

The reports generally contained entities and topics that genuinely appear in the source. However, they frequently converted policy objectives, recommendations and possible benefits into established outcomes.

Examples include claims that PV measures:

- ensure long-term profitability,
- increase grid resilience,
- create rural economic resilience,
- guarantee economic growth,
- directly produce public acceptance,
- shape continental electricity markets.

The source discusses many of these effects as goals, possibilities or expected benefits rather than empirically demonstrated results.

### 2. Unsupported leadership language

Several reports describe Austria as:

- a European frontrunner,
- a model for other countries,
- being at the forefront of the European energy transition,
- demonstrating international technological leadership.

These formulations are stronger than the evidence provided by the Austrian Photovoltaic Strategy.

### 3. Institutional roles were expanded

Some reports assigned organizations broader roles than the source establishes.

For example, a bibliographic reference to ÖGUT was transformed into claims that ÖGUT had a central operational role and that its methods had been formally evaluated and adopted by government.

Similarly, descriptions of BMK, E-Control and EU institutions were expanded into complete governance and enforcement narratives.

### 4. Intended measures were presented as completed implementation

The source is a strategy document and therefore contains many planned or recommended measures. Community reports sometimes describe these measures as already implemented, successful or institutionally established.

This creates confusion between:

- what Austria plans to do,
- what the strategy recommends,
- what has legally been adopted,
- and what has already produced measurable effects.

### 5. Errors propagated from earlier graph stages

Every reviewed report inherited at least one weakness from the extracted entities or relationships.

Examples include:

- PV installations represented as organizations,
- policy documents represented as events,
- aliases maintained as separate entities,
- inferred relationships treated as explicit facts,
- overly broad entity descriptions.

The community-report stage does not correct these problems. Instead, it produces fluent narratives from them, which can make the underlying errors more difficult to notice.

## Representative evidence chain

### Example 1 — Austrian renewable leadership

**Source evidence:** The strategy presents objectives and measures for expanding photovoltaics in Austria.

**GraphRAG output:** Austria is described as a European frontrunner, international leader and model for other countries.

**Problem:** Strategic ambition was transformed into demonstrated international leadership.

**Likely pipeline stage:** Community-report generation, combined with generalized entity and relationship descriptions.

### Example 2 — Agri-PV impacts

**Source evidence:** Agri-PV can support dual land use by combining agricultural production with photovoltaic electricity generation.

**GraphRAG output:** Agri-PV is said to guarantee food security, rural economic resilience, market diversity and strong public acceptance.

**Problem:** Potential advantages were presented as established causal outcomes.

**Likely pipeline stage:** Relationship extraction and community-report generalization.

### Example 3 — ÖGUT's role

**Source evidence:** An ÖGUT publication concerning methods for social innovation is referenced.

**GraphRAG output:** ÖGUT is presented as having a central implementation role, with formally evaluated and government-adopted methods.

**Problem:** A bibliographic reference was transformed into an active institutional and decision-making role.

**Likely pipeline stage:** Entity-description summarization followed by community-report generation.

## Conclusion

The community reports are useful as exploratory summaries, but they are not sufficiently reliable to serve as authoritative knowledge-graph facts.

The major problem is not that the reports are completely unrelated to the source. Most contain genuine source topics. The problem is that they combine those topics into confident narratives containing unsupported causality, institutional roles, implementation status and leadership claims.

Community reports should therefore be treated as:

- retrieval and exploration aids,
- secondary LLM-generated interpretations,
- material requiring source verification,

and not as direct evidence for the controlled solar-market knowledge graph.

For the next GraphRAG experiment, community reports should remain available for question answering, but claims used in the controlled KG should be traced back to text units and the original document.